# Практика: Бустинг — от теории к CatBoost-мастерству

**Подготовка к Всероссийской олимпиаде по ИИ, Сириус 2026**


---

### Содержание

1. **Часть 1.** Градиентный бустинг — теория и реализация с нуля
2. **Часть 2.** AdaBoost — экспоненциальный loss и взвешивание
3. **Часть 3.** XGBoost — аппроксимация Тейлора и регуляризация
4. **Часть 4.** LightGBM — leaf-wise, GOSS, EFB
5. **Часть 5.** CatBoost — глубокий разбор внутреннего устройства
6. **Часть 6.** Сравнение бустингов на реальных данных
7. **Часть 7.** CatBoost: практическое мастерство для олимпиады
8. **Часть 8.** Итоговые задания и вопросы

---


<img src="https://encrypted-tbn3.gstatic.com/images?q=tbn:ANd9GcQ-l10K9iXY2_gdl6LtxEurkKiADJYCzg1q0NE5mIqGxAl090hI" width="500">

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from time import time

from sklearn.datasets import load_digits, fetch_california_housing
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    GradientBoostingRegressor, GradientBoostingClassifier,
    RandomForestClassifier, RandomForestRegressor,
    AdaBoostClassifier
)
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_curve
)
from sklearn.model_selection import (
    cross_val_score, cross_validate, GridSearchCV,
    train_test_split
)
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 13

---

## Часть 1. Градиентный бустинг — теория и реализация с нуля

### 1.1. Общая идея

**Бустинг** строит ансамбль **последовательно**. Каждая следующая модель исправляет ошибки предыдущих.

В отличие от бэггинга (снижает Variance), бустинг последовательно снижает **Bias**.

### 1.2. Градиентный бустинг (Friedman, 2001)

Строим аддитивную модель:

$$
F_M(x) = F_0(x) + \sum_{m=1}^{M} \eta \cdot h_m(x)
$$

где $h_m$ — базовые модели (неглубокие деревья), $\eta$ — learning rate.

### 1.3. Вывод алгоритма

Хотим минимизировать эмпирический риск:

$$
R(F) = \sum_{i=1}^N L(y_i, F(x_i))
$$

На шаге $m$ мы фиксируем $F_{m-1}$ и ищем $h_m$, минимизирующее:

$$
h_m = \arg\min_h \sum_{i=1}^N L(y_i, F_{m-1}(x_i) + h(x_i))
$$

**Ключевая идея:** рассмотрим $F(x_i)$ как «параметры» и выполним один шаг **градиентного спуска** в функциональном пространстве:

$$
F_m(x_i) = F_{m-1}(x_i) - \eta \cdot \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\bigg|_{F = F_{m-1}}
$$

Обозначим **псевдо-остатки** (антиградиент):

$$
r_{im} = -\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\bigg|_{F = F_{m-1}}
$$

Обучаем $h_m$ **аппроксимировать** эти псевдо-остатки: $h_m \approx \{r_{im}\}_{i=1}^N$.

### 1.4. Для MSE loss

$$
L(y, f) = \frac{1}{2}(y - f)^2 \implies r_{im} = -\frac{\partial}{\partial f}\frac{1}{2}(y_i - f)^2\bigg|_{f=F_{m-1}(x_i)} = y_i - F_{m-1}(x_i)
$$

То есть для MSE псевдо-остатки — это просто **остатки**.

### 1.5. Для Log-loss (бинарная классификация)

$$
L(y, f) = -y\log\sigma(f) - (1-y)\log(1-\sigma(f)), \quad \sigma(f) = \frac{1}{1+e^{-f}}
$$

$$
r_{im} = y_i - \sigma(F_{m-1}(x_i))
$$

### 1.6. Для MAE loss

$$
L(y, f) = |y - f| \implies r_{im} = \text{sign}(y_i - F_{m-1}(x_i))
$$

### 1.7. Полный алгоритм

1. $F_0(x) = \arg\min_c \sum_{i=1}^N L(y_i, c)$ (для MSE: $F_0 = \bar{y}$)
2. Для $m = 1, \ldots, M$:
   - $r_{im} = -\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\big|_{F_{m-1}}$
   - Обучить $h_m$ на $(x_i, r_{im})$
   - $F_m = F_{m-1} + \eta \cdot h_m$

### 1.8. Learning Rate (Shrinkage)

- $\eta \in (0, 1]$ масштабирует вклад каждого дерева
- Маленький $\eta$ → больше деревьев, но лучше обобщение
- Аналогия: $\eta$ в GD — размер шага. Маленький шаг → аккуратнее приходим к минимуму
- Типичные значения: $\eta \in [0.01, 0.3]$

### 1.9. Стохастический градиентный бустинг

Friedman (2002): на каждом шаге используем случайную подвыборку объектов (subsample) и/или признаков. Это:
- Снижает Variance (аналогично RF)
- Ускоряет обучение
- Типичные значения: `subsample ∈ [0.5, 0.8]`, `colsample ∈ [0.5, 1.0]`

### Задание 1.1. Реализация градиентного бустинга для регрессии (MSE)

Реализуйте класс `GradientBoostingMSE` с методами `fit` и `predict`.

- `predict` должен принимать необязательный аргумент `n_trees`, чтобы можно было получить предсказание после первых $n$ деревьев.

In [ ]:
class GradientBoostingMSE:
    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.trees = []
        self.initial_prediction = None

    def fit(self, X, y):
        # YOUR CODE HERE
        pass

    def predict(self, X, n_trees=None):
        # YOUR CODE HERE
        pass

### Задание 1.2. Тест на синтетических данных

Функция: $f(x) = \exp(-x^2) + 1.5 \exp(-(x-2)^2)$

1. Сгенерируйте train (150 точек, noise=0.1) и test (1000 точек, noise=0)
2. Визуализируйте данные
3. Обучите `DecisionTreeRegressor(max_depth=5)` — предскажите и визуализируйте
4. Обучите ваш `GradientBoostingMSE(n_estimators=100, max_depth=3, lr=0.1)` — визуализируйте
5. Сравните MSE на тесте

In [ ]:
def generate(n_samples, noise=0.1):
    x = np.linspace(-2, 5, n_samples)
    y = np.exp(-x**2) + 1.5 * np.exp(-(x - 2)**2) + np.random.normal(0, noise, n_samples)
    return x.reshape(-1, 1), y

X_train_synth, y_train_synth = generate(150, noise=0.1)
X_test_synth, y_test_synth = generate(1000, noise=0.0)

# YOUR CODE HERE

### Задание 1.3. Эволюция предсказаний

Обучите бустинг на 150 деревьях. Визуализируйте предсказания для `n_trees` $\in \{1, 3, 10, 30, 50, 80, 100, 150\}$ (subplots 2×4). На каждом графике покажите $f(x)$ (красная), предсказание (синяя), точки train (серые).

In [ ]:
# YOUR CODE HERE

### Задание 1.4. Влияние learning rate

Постройте MSE(test) от числа деревьев (1–300) для $\eta \in \{0.01, 0.05, 0.1, 0.3, 1.0\}$. Все 5 линий на одном графике, ось Y — log scale.

In [ ]:
# YOUR CODE HERE

**Вопрос 1.1.** Что происходит при $\eta = 1.0$? Почему MSE на тесте может **расти** при добавлении деревьев?

*Ваш ответ:*

**Вопрос 1.2.** Как связаны $\eta$ и оптимальное число деревьев $M$? Можно ли утверждать, что $\eta \cdot M \approx \text{const}$?

*Ваш ответ:*

**Вопрос 1.3.** Бустинг снижает bias. Но может ли он при этом увеличить variance? Когда это происходит?

*Ваш ответ:*

### Задание 1.5. Бустинг на California Housing

1. Загрузите `fetch_california_housing`, split train/test (80/20)
2. Обучите `DecisionTreeRegressor(max_depth=5)` — R² на train и test
3. Обучите ваш `GradientBoostingMSE(n_estimators=200, max_depth=4, lr=0.1)` — R² на train и test
4. Сравните со sklearn `GradientBoostingRegressor` с теми же параметрами

In [ ]:
# YOUR CODE HERE

### Задание 1.6. MSE/R² в зависимости от числа деревьев

Для California Housing постройте:
- MSE на train и test от числа деревьев (50, 70, 90, ..., 250)
- R² на train и test от числа деревьев

Используйте `cross_validate` из sklearn.

In [ ]:
# YOUR CODE HERE

### Задание 1.7. Бустинг vs Random Forest на digits

Сравните `GradientBoostingClassifier` и `RandomForestClassifier` на `digits` (7-fold CV, accuracy). Кто лучше?

In [ ]:
# YOUR CODE HERE

---

## Часть 2. AdaBoost

### 2.1. Теория

**AdaBoost** (Freund & Schapire, 1996) — каждый объект получает вес; «сложные» объекты получают больший вес.

### 2.2. Алгоритм (бинарная классификация, $y_i \in \{-1, +1\}$)

1. $w_i^{(1)} = \frac{1}{N}$ для всех $i$

2. Для $m = 1, \ldots, M$:

   a) Обучить $h_m$ на $(X, y)$ с весами $w^{(m)}$
   
   b) Взвешенная ошибка: $\varepsilon_m = \frac{\sum_{i} w_i^{(m)} [h_m(x_i) \neq y_i]}{\sum_{i} w_i^{(m)}}$
   
   c) Вес модели: $\alpha_m = \frac{1}{2} \ln \frac{1 - \varepsilon_m}{\varepsilon_m}$
   
   d) Обновление: $w_i^{(m+1)} = w_i^{(m)} \exp(-\alpha_m y_i h_m(x_i))$
   
   e) Нормировка: $w_i^{(m+1)} \leftarrow \frac{w_i^{(m+1)}}{\sum_j w_j^{(m+1)}}$

3. Итого: $F(x) = \text{sign}\!\left(\sum_{m=1}^M \alpha_m h_m(x)\right)$

### 2.3. Связь с экспоненциальным loss

AdaBoost — это **градиентный бустинг с экспоненциальным loss:**

$$
L(y, f) = \exp(-y \cdot f)
$$

**Вывод $\alpha_m$:** минимизируем $\sum_i w_i \exp(-\alpha \, y_i h(x_i))$ по $\alpha$:

$$
= e^{-\alpha}\sum_{i: h(x_i)=y_i} w_i + e^{\alpha}\sum_{i: h(x_i) \neq y_i} w_i
$$

Берём производную по $\alpha$, приравниваем к нулю:

$$
\alpha^* = \frac{1}{2}\ln\frac{1-\varepsilon}{\varepsilon}
$$

### 2.4. Чувствительность к выбросам

Экспоненциальный loss **экспоненциально** растёт при больших ошибках → AdaBoost чувствителен к шуму и выбросам.

Сравнение loss-функций при $y=1$:

| Loss | $L(1, f)$ | Поведение при $f \to -\infty$ |
|------|-----------|------------------------------|
| Exp | $e^{-f}$ | $\to +\infty$ (экспоненциально) |
| Log | $\log(1+e^{-f})$ | $\to +\infty$ (линейно) |
| Hinge | $\max(0, 1-f)$ | $\to +\infty$ (линейно) |

### Задание 2.1. Реализация AdaBoost

Реализуйте `MyAdaBoost` для бинарной классификации ($y \in \{-1, +1\}$). Используйте `DecisionTreeClassifier(max_depth=1)` (stump) и `sample_weight`.

In [ ]:
class MyAdaBoost:
    def __init__(self, n_estimators=50, max_depth=1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.estimators = []
        self.alphas = []

    def fit(self, X, y):
        # YOUR CODE HERE
        pass

    def predict(self, X):
        # YOUR CODE HERE
        pass

### Задание 2.2. Тестирование AdaBoost

На `digits` (бинарная: 3 vs 8) сравните:
1. Ваш `MyAdaBoost`
2. `AdaBoostClassifier` из sklearn
3. `GradientBoostingClassifier` из sklearn

CV accuracy (5-fold).

In [ ]:
digits = load_digits()
mask = np.isin(digits.target, [3, 8])
X_bin = digits.data[mask]
y_bin = digits.target[mask]
y_bin_ada = np.where(y_bin == 3, -1, 1)

# YOUR CODE HERE

### Задание 2.3. Визуализация весов AdaBoost

Обучите AdaBoost (20 итераций) на make_moons (100 точек, noise=0.3). На каждой итерации {1, 5, 10, 20} нарисуйте scatter plot, где размер точки пропорционален весу $w_i$.

In [ ]:
# YOUR CODE HERE

**Вопрос 2.1.** Почему AdaBoost чаще использует stumps (деревья глубины 1)? Что будет, если использовать глубокие деревья?

*Ваш ответ:*

---

## Часть 3. XGBoost

### 3.1. Регуляризация в целевой функции

XGBoost (Chen & Guestrin, 2016) оптимизирует:

$$
\mathcal{L} = \sum_{i=1}^N L(y_i, \hat{y}_i) + \sum_{m=1}^M \Omega(h_m)
$$

$$
\Omega(h) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2
$$

где $T$ — число листьев, $w_j$ — значение в листе $j$, $\gamma$ — штраф за сложность, $\lambda$ — L2-регуляризация.

### 3.2. Аппроксимация Тейлора второго порядка

На шаге $m$, разложим loss:

$$
L(y_i, \hat{y}_i^{(m-1)} + h_m(x_i)) \approx L(y_i, \hat{y}_i^{(m-1)}) + g_i h_m(x_i) + \frac{1}{2} h_i h_m^2(x_i)
$$

где:

$$
g_i = \frac{\partial L(y_i, \hat{y})}{\partial \hat{y}}\bigg|_{\hat{y}^{(m-1)}} \quad \text{(градиент)}, \qquad h_i = \frac{\partial^2 L(y_i, \hat{y})}{\partial \hat{y}^2}\bigg|_{\hat{y}^{(m-1)}} \quad \text{(гессиан)}
$$

### 3.3. Оптимальное значение в листе

Для листа $j$ с объектами $I_j$, минимизация квадратичной аппроксимации + регуляризатор даёт:

$$
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}
$$

### 3.4. Критерий разбиения (Gain)

$$
\text{Gain} = \frac{1}{2}\left[\frac{\left(\sum_{i \in I_L} g_i\right)^2}{\sum_{i \in I_L} h_i + \lambda} + \frac{\left(\sum_{i \in I_R} g_i\right)^2}{\sum_{i \in I_R} h_i + \lambda} - \frac{\left(\sum_{i \in I} g_i\right)^2}{\sum_{i \in I} h_i + \lambda}\right] - \gamma
$$

Если $\text{Gain} < 0$, разбиение не выполняется (pre-pruning через $\gamma$).

### 3.5. Другие фичи

- Column subsampling (как в RF)
- Approximate split finding (квантильные бины)
- Sparsity-aware split (обработка пропусков: default direction)
- Параллелизация по признакам (не по деревьям!)

In [ ]:
# !pip install xgboost lightgbm catboost --quiet
import xgboost as xgb

### Задание 3.1. XGBoost на telecom_data

1. Загрузите `telecom_data.csv`
2. Предобработка: Churn → int, закодируйте `International plan`, `Voice mail plan`, `State` (LabelEncoder)
3. Split train/test (80/20, stratify)
4. Обучите `XGBClassifier(n_estimators=200, max_depth=5, lr=0.1)`
5. Выведите Accuracy, F1, ROC-AUC на тесте

In [ ]:
# YOUR CODE HERE

### Задание 3.2. Пошаговый GridSearch для XGBoost

Подберите гиперпараметры **последовательно**:
1. `n_estimators` ∈ {100, 200, 300, 500}, `learning_rate` ∈ {0.01, 0.05, 0.1, 0.2}
2. С лучшими — `max_depth` ∈ {3, 5, 7, 9}, `min_child_weight` ∈ {1, 3, 5}
3. С лучшими — `subsample` ∈ {0.6, 0.8, 1.0}, `colsample_bytree` ∈ {0.6, 0.8, 1.0}
4. С лучшими — `reg_alpha` ∈ {0, 0.1, 1}, `reg_lambda` ∈ {1, 5, 10}

Для каждого шага выведите лучшие параметры и лучший CV score.

In [ ]:
# Шаг 1
# YOUR CODE HERE

In [ ]:
# Шаг 2
# YOUR CODE HERE

In [ ]:
# Шаг 3
# YOUR CODE HERE

In [ ]:
# Шаг 4
# YOUR CODE HERE

**Вопрос 3.1.** Зачем в XGBoost используется гессиан (информация второго порядка)? Что это даёт по сравнению с обычным GBM?

*Ваш ответ:*

**Вопрос 3.2.** Как параметр $\gamma$ в XGBoost связан с pruning? Почему это лучше, чем post-pruning?

*Ваш ответ:*

---

## Часть 4. LightGBM

### 4.1. Leaf-wise vs Level-wise

- **Level-wise** (XGBoost, sklearn): на каждом уровне разбиваются **все** листья → сбалансированное дерево
- **Leaf-wise** (LightGBM): разбивается лист с **максимальным $\Delta$ loss** → несбалансированное, но более точное дерево

Leaf-wise быстрее сходится, но может переобучаться → контролируется `num_leaves` и `max_depth`.

### 4.2. GOSS (Gradient-based One-Side Sampling)

Объекты с большими $|g_i|$ (плохо предсказанные) важнее:
1. Отсортировать объекты по $|g_i|$
2. Взять top-$a$ объектов (большие градиенты)
3. Случайно выбрать $b$ объектов из остальных, домножить их градиенты на $\frac{1-a}{b}$

### 4.3. EFB (Exclusive Feature Bundling)

Взаимоисключающие разреженные признаки объединяются в bundles → уменьшение эффективного числа признаков.

### 4.4. Гиперпараметры LightGBM

| Параметр | Описание | Типичные значения |
|----------|----------|-------------------|
| `num_leaves` | Макс. число листьев | 31–255 |
| `max_depth` | Ограничение глубины | -1 (без) |
| `learning_rate` | Shrinkage | 0.01–0.3 |
| `n_estimators` | Число деревьев | 100–10000 |
| `min_child_samples` | Мин. объектов в листе | 5–100 |
| `subsample` | Доля объектов | 0.5–1.0 |
| `colsample_bytree` | Доля признаков | 0.5–1.0 |
| `reg_alpha` | L1 | 0–10 |
| `reg_lambda` | L2 | 0–10 |

### Задание 4.1. LightGBM на telecom_data

Обучите `LGBMClassifier` на тех же данных. Выведите Accuracy, F1, ROC-AUC.

In [ ]:
import lightgbm as lgb

# YOUR CODE HERE

**Вопрос 4.1.** При каких размерах данных LightGBM существенно быстрее XGBoost? Почему?

*Ваш ответ:*

---

## Часть 5. CatBoost — глубокий разбор внутреннего устройства

### 5.1. Три ключевых инновации

**CatBoost** (Prokhorenkova et al., 2018, Yandex):
1. **Ordered Target Statistics** — обработка категориальных признаков
2. **Ordered Boosting** — борьба с prediction shift
3. **Oblivious Trees** — симметричные деревья

### 5.2. Проблема Target Leakage

Наивное кодирование категории $c_i$ через mean target:

$$
\hat{x}_i = \frac{\sum_{j=1}^{N} [c_j = c_i] \cdot y_j}{\sum_{j=1}^{N} [c_j = c_i]}
$$

**Проблема:** $y_i$ участвует в вычислении $\hat{x}_i$ → **target leakage** → переобучение.

### 5.3. Ordered Target Statistics (CatBoost)

Фиксируем случайную перестановку $\sigma$ объектов. Для объекта $\sigma(i)$ статистика считается **только по предыдущим**:

$$
\hat{x}_{\sigma(i)} = \frac{\sum_{j=1}^{i-1} [c_{\sigma(j)} = c_{\sigma(i)}] \cdot y_{\sigma(j)} + a \cdot p}{\sum_{j=1}^{i-1} [c_{\sigma(j)} = c_{\sigma(i)}] + a}
$$

где $a > 0$ — prior weight, $p$ — prior (mean target по всей выборке).

**Зачем prior:** для первых объектов мало данных → $\hat{x} \approx p$ (стабильная оценка).

### 5.4. Prediction Shift

В обычном GBM остатки $r_i = y_i - F_{m-1}(x_i)$ считаются моделью $F_{m-1}$, обученной **на $x_i$**.

Это значит, что $r_i$ **смещены вниз** (модель уже «видела» $x_i$). Формально:

$$
\mathbb{E}[r_i \mid x_i] \neq \mathbb{E}[r \mid x] \quad (\text{для нового } x)
$$

Дерево $h_m$, обученное на смещённых остатках, даёт **смещённые** предсказания.

### 5.5. Ordered Boosting

Для каждого $x_{\sigma(i)}$ предсказание $F_{m-1}$ считается моделью, обученной **только на** $\{x_{\sigma(1)}, \ldots, x_{\sigma(i-1)}\}$.

На практике: $O(\log N)$ моделей разного размера (mode-based approximation).

### 5.6. Oblivious Decision Trees

На **каждом уровне** дерева одно и то же разбиение для **всех** узлов.

Дерево глубины $d$: ровно $2^d$ листьев. Каждый объект → бинарный вектор длины $d$ → лист.

**Преимущества:**
- Быстрый inference: $O(d)$ сравнений (вместо path-dependent)
- Регуляризация (ограничение сложности)
- Cache-friendly (быстрые предсказания на CPU)

### 5.7. Комбинации категориальных признаков

На каждом уровне дерева CatBoost может создать **новый** категориальный признак как комбинацию существующих (feature interactions).

### 5.8. Гиперпараметры CatBoost

| Параметр | Описание | Типичные значения |
|----------|----------|-------------------|
| `iterations` | Число деревьев | 100–10000 |
| `depth` | Глубина oblivious trees | 4–10 |
| `learning_rate` | Shrinkage | 0.01–0.3 |
| `l2_leaf_reg` | L2-регуляризация в листьях | 1–10 |
| `random_strength` | Случайность при выборе разбиения | 0–10 |
| `bagging_temperature` | Bayesian bootstrap intensity | 0–10 |
| `border_count` | Число бинов для числовых признаков | 32–254 |
| `cat_features` | Индексы категориальных признаков | — |
| `grow_policy` | SymmetricTree / Depthwise / Lossguide | SymmetricTree |
| `boosting_type` | Ordered / Plain | Ordered |

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor, Pool, cv as catboost_cv

### Задание 5.1. CatBoost с cat_features на telecom_data

1. Загрузите `telecom_data.csv` **без** кодирования категорий
2. Определите `cat_features` (State, International plan, Voice mail plan, Area code)
3. Обучите `CatBoostClassifier(iterations=500, depth=6, lr=0.1)` с `cat_features`
4. Используйте `eval_set` и `early_stopping_rounds=50`
5. Выведите Accuracy, F1, ROC-AUC на тесте

In [ ]:
# YOUR CODE HERE

### Задание 5.2. Сравнение: с cat_features vs без (LabelEncoder)

Обучите CatBoost **без** `cat_features` (закодировав категории вручную через LabelEncoder). Сравните F1.

In [ ]:
# YOUR CODE HERE

**Вопрос 5.1.** Почему CatBoost с `cat_features` может работать лучше? Объясните через Ordered Target Statistics.

*Ваш ответ:*

**Вопрос 5.2.** Как параметр `a` (prior weight) в Ordered Target Statistics влияет на кодирование? Что будет при $a=0$? При $a \to \infty$?

*Ваш ответ:*

---

## Часть 6. Сравнение всех бустингов на реальных данных

### Задание 6.1. Сравнение на telecom_data

Обучите **все модели** с дефолтными параметрами (200 деревьев, lr=0.1). Выведите таблицу: Model, Accuracy, F1, ROC-AUC, Train time.

Модели:
- sklearn GradientBoosting
- AdaBoost
- XGBoost
- LightGBM
- CatBoost
- Random Forest (для сравнения)

In [ ]:
# YOUR CODE HERE

### Задание 6.2. Визуализация сравнения

Постройте 3 горизонтальных bar chart (subplots 1×3): Accuracy, F1, ROC-AUC.

In [ ]:
# YOUR CODE HERE

### 6.3. flight_delays — тест на категориальных данных

Загрузите `flight_delays_train.csv`. Предобработка:
- `dep_delayed_15min`: 'Y' → 1, else 0
- `Month`, `DayofMonth`, `DayOfWeek`: убрать 'c-', привести к int
- `UniqueCarrier`, `Origin`, `Dest` — категориальные

In [ ]:
df_flights = pd.read_csv('flight_delays_train.csv')

df_flights['dep_delayed_15min'] = (df_flights['dep_delayed_15min'] == 'Y').astype(int)
for col in ['Month', 'DayofMonth', 'DayOfWeek']:
    df_flights[col] = df_flights[col].str.replace('c-', '').astype(int)

y_fl = df_flights['dep_delayed_15min'].values
X_fl = df_flights.drop('dep_delayed_15min', axis=1)

cat_cols_fl = ['UniqueCarrier', 'Origin', 'Dest']
cat_idx_fl = [X_fl.columns.get_loc(c) for c in cat_cols_fl]

X_fl_train, X_fl_test, y_fl_train, y_fl_test = train_test_split(
    X_fl, y_fl, test_size=0.2, random_state=42, stratify=y_fl
)

print(f'Train: {X_fl_train.shape}, Test: {X_fl_test.shape}')
print(f'Delay rate: {y_fl.mean():.3f}')
print(f'Категориальные: {cat_cols_fl}')

### Задание 6.3. Сравнение на flight_delays

Обучите и сравните:
1. **XGBoost** — нужен one-hot encoding (`pd.get_dummies`) для категориальных
2. **LightGBM** — аналогично (или нативная поддержка категорий)
3. **CatBoost** — с `cat_features`

Выведите таблицу: Model, Accuracy, ROC-AUC, Train time.

In [ ]:
# YOUR CODE HERE

**Вопрос 6.1.** Какая модель показала лучший ROC-AUC? Почему CatBoost может выигрывать на данных с категориями?

*Ваш ответ:*

**Вопрос 6.2.** Какая модель обучилась быстрее? Как соотносятся скорость и качество?

*Ваш ответ:*

**Вопрос 6.3.** Почему one-hot encoding для Origin/Dest (сотни уникальных значений) — плохая идея? Какие альтернативы?

*Ваш ответ:*

---

## Часть 7. CatBoost: практическое мастерство для олимпиады

### 7.1. Pool — контейнер данных

Создайте `Pool` для train и test с указанием `cat_features`.

In [ ]:
# YOUR CODE HERE: создайте train_pool и test_pool

### Задание 7.1. Early Stopping и кривые обучения

1. Обучите CatBoost с `iterations=2000, lr=0.05, eval_metric='AUC', early_stopping_rounds=100`
2. Выведите лучшее число итераций и лучший AUC
3. Постройте learning curves: Logloss (train и val) и AUC (val) от итерации. Отметьте best iteration вертикальной линией.

In [ ]:
# YOUR CODE HERE

### Задание 7.2. Feature Importance

1. Выведите `get_feature_importance(prettified=True)`
2. Визуализируйте горизонтальным bar chart

In [ ]:
# YOUR CODE HERE

### Задание 7.3. SHAP Values

1. Вычислите `get_feature_importance(pool, type='ShapValues')`
2. Вычислите mean |SHAP| по признакам
3. Визуализируйте top-10 признаков

In [ ]:
# YOUR CODE HERE

### Задание 7.4 (сложное). Пошаговый подбор параметров CatBoost

На `flight_delays`:

**Шаг 1.** `depth` ∈ {4, 6, 8, 10}, `learning_rate` ∈ {0.01, 0.05, 0.1, 0.2} (с early_stopping)

**Шаг 2.** С лучшими — `l2_leaf_reg` ∈ {1, 3, 5, 7, 10, 15, 20}

**Шаг 3.** С лучшими — `random_strength` ∈ {0, 1, 5, 10}, `bagging_temperature` ∈ {0, 1, 5, 10}

**Шаг 4.** Финальная модель с найденными параметрами

In [ ]:
# Шаг 1
# YOUR CODE HERE

In [ ]:
# Шаг 2
# YOUR CODE HERE

In [ ]:
# Шаг 3
# YOUR CODE HERE

In [ ]:
# Шаг 4: финальная модель
# YOUR CODE HERE

### Задание 7.5 (сложное). Сравнение grow_policy

Сравните `SymmetricTree`, `Depthwise`, `Lossguide` на flight_delays. Для Depthwise/Lossguide используйте `max_leaves=64`.

Выведите таблицу: Policy, AUC, Best Iteration, Time.

In [ ]:
# YOUR CODE HERE

**Вопрос 7.1.** Какая `grow_policy` показала лучший результат? Объясните разницу между тремя стратегиями.

*Ваш ответ:*

### Задание 7.6 (сложное). Ordered vs Plain boosting

Сравните `boosting_type='Ordered'` и `boosting_type='Plain'`. Выведите AUC и best iteration.

In [ ]:
# YOUR CODE HERE

**Вопрос 7.2.** Что такое prediction shift? Как Ordered boosting с ним борется? В каких случаях Plain достаточен?

*Ваш ответ:*

### Задание 7.7. Confusion Matrix и оптимальный порог

1. Постройте confusion matrix для лучшей модели
2. Постройте Precision, Recall, F1 от порога (0–1)
3. Найдите оптимальный порог для F1
4. Сравните F1 при threshold=0.5 и при оптимальном

In [ ]:
# YOUR CODE HERE

### Задание 7.8. CatBoost CV (встроенная кросс-валидация)

Используйте `catboost.cv` для подбора числа итераций. Постройте график AUC ± std от итерации.

In [ ]:
# YOUR CODE HERE

### Задание 7.9. Сохранение и загрузка модели

Сохраните лучшую модель в `catboost_best.cbm`. Загрузите и проверьте, что предсказания совпадают.

In [ ]:
# YOUR CODE HERE

---

## Часть 8. Итоговые задания и тесты

### Тест 8.1. Теоретические вопросы

**1.** В градиентном бустинге каждое дерево обучается на:
- a) Исходных метках $y_i$
- b) Антиградиенте функции потерь (псевдо-остатках)
- c) Случайной подвыборке данных
- d) Взвешенных данных

**2.** Какое утверждение о learning rate в бустинге **ВЕРНО**?
- a) Большой lr всегда лучше — быстрее сходится
- b) Маленький lr требует больше деревьев, но лучше обобщает
- c) lr не влияет на качество, только на скорость
- d) Оптимальный lr всегда = 1.0

**3.** XGBoost отличается от обычного GBM тем, что:
- a) Использует информацию второго порядка (гессиан)
- b) Не поддерживает регуляризацию
- c) Строит только stumps
- d) Использует бэггинг

**4.** CatBoost Ordered Target Statistics нужен для:
- a) Ускорения обучения
- b) Предотвращения target leakage при кодировании категорий
- c) Параллелизации
- d) Оптимизации памяти

**5.** Oblivious Trees в CatBoost означают:
- a) Деревья строятся случайно
- b) На каждом уровне одно разбиение для всех узлов
- c) Неограниченная глубина
- d) Нет категориальных признаков

**6.** Leaf-wise (LightGBM) vs Level-wise (XGBoost):
- a) Leaf-wise всегда лучше
- b) Leaf-wise может давать более точные, но менее устойчивые деревья
- c) Level-wise строит более компактные деревья
- d) Leaf-wise не может переобучаться

**7.** Псевдо-остатки для MAE loss ($L = |y - f|$) равны:
- a) $y - f$
- b) $\text{sign}(y - f)$
- c) $(y - f)^2$
- d) $\exp(-yf)$

**8.** Бустинг в основном снижает:
- a) Variance (как бэггинг)
- b) Bias
- c) Noise
- d) И Bias, и Variance одинаково

### Задание 8.1 (олимпиадное). Максимальный ROC-AUC на flight_delays

Добейтесь максимального ROC-AUC. Используйте всё: `cat_features`, early stopping, подбор параметров, CV.

**Цель:** ROC-AUC > 0.74

In [ ]:
# YOUR CODE HERE

### Задание 8.2 (олимпиадное). Feature Engineering

Создайте новые признаки и проверьте, улучшают ли они AUC:
1. `Carrier_Origin` = `UniqueCarrier` + '_' + `Origin` (новый категориальный)
2. `DepHour` = `DepTime // 100` (час вылета)
3. `is_weekend` = `DayOfWeek` ∈ {6, 7}
4. `is_evening` = `DepTime` > 1800

Сравните AUC с/без новых признаков.

In [ ]:
# YOUR CODE HERE

### Задание 8.3 (повышенной сложности). Реализация Ordered Target Statistics

Реализуйте функцию `ordered_target_statistics(categories, targets, prior, alpha)` с нуля. Протестируйте на примере.

In [ ]:
def ordered_target_statistics(categories, targets, prior=None, alpha=1.0, random_state=42):
    """Ordered Target Statistics (как в CatBoost)."""
    # YOUR CODE HERE
    pass

# Тест:
# cats = np.array(['A', 'B', 'A', 'A', 'B', 'A', 'B', 'B'])
# tgts = np.array([1, 0, 1, 0, 1, 1, 0, 1])
# print(ordered_target_statistics(cats, tgts))

### Задание 8.4 (повышенной сложности). Градиентный бустинг для классификации

Реализуйте градиентный бустинг с **log-loss** (бинарная классификация). Псевдо-остатки: $r_i = y_i - \sigma(F_{m-1}(x_i))$. Протестируйте на make_moons.

In [ ]:
# YOUR CODE HERE

### Задание 8.5. Открытые вопросы на интерпретацию

**Вопрос 8.1.** Вы решаете олимпиадную задачу: 10 числовых + 5 категориальных признаков, 50K объектов, 15% положительного класса. Какой бустинг выберете? Какие параметры будете подбирать первыми?

*Ваш ответ:*

**Вопрос 8.2.** Вы обучили CatBoost (AUC=0.85) и XGBoost (AUC=0.87) на local CV. На public leaderboard: CatBoost — 0.84, XGBoost — 0.82. Что могло произойти?

*Ваш ответ:*

**Вопрос 8.3.** Как бустинг влияет на Bias и Variance по сравнению с бэггингом? Почему бустинг может переобучаться при увеличении числа деревьев, а RF — практически нет?

*Ваш ответ:*

**Вопрос 8.4.** Градиентный бустинг — это «градиентный спуск в функциональном пространстве». Объясните эту аналогию: что является «параметрами»? Что — функцией потерь? Что — градиентом?

*Ваш ответ:*

**Вопрос 8.5.** Вам дан dataset с 1000 числовыми признаками и 200 объектами. Какой бустинг (если вообще) вы выберете? Какие параметры регуляризации критичны?

*Ваш ответ:*

---

### Резюме и Cheatsheet для олимпиады

#### Когда какой бустинг?

| Ситуация | Рекомендация |
|----------|-------------|
| Много категориальных признаков | **CatBoost** (Ordered Target Statistics) |
| Большие данные (>1M строк) | **LightGBM** (GOSS + EFB) |
| Данные с пропусками | **XGBoost** или **LightGBM** |
| Нужна максимальная точность | **CatBoost** + тщательный подбор |
| Быстрый baseline | **LightGBM** с дефолтами |
| Олимпиадные задачи | **CatBoost** (устойчив, обрабатывает категории) |

#### Порядок подбора гиперпараметров CatBoost

1. `iterations` + `learning_rate` (с early_stopping)
2. `depth` (4–10)
3. `l2_leaf_reg` (1–20)
4. `random_strength` + `bagging_temperature`
5. `border_count` (64–254)

#### Типичные ошибки

- Не указать `cat_features` → CatBoost не использует Ordered TS
- Много итераций без early stopping → overfitting
- Один split на train/test → нечестная оценка
- Забыть `random_seed` → невоспроизводимость
- OHE для высококардинальных признаков → раздувание размерности

---

**Удачи на олимпиаде!**